# Phishing Email Detector

In [5]:
#importing the dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from email.utils import parseaddr
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [20]:
import feature_extractor

## Loading the dataset 

In [3]:
df=pd.read_csv("Nazario_5.csv")

In [4]:
df.head()

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr\nPW: bnaweb22\n\n\n ...,0,"['http://web.bna.com', 'http://pubs.bna.com/ip..."
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,[]
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,['http://eastpower.dev.corp.enron.com/summary/...
3,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>","""Grass, John"" <John.Grass@ENRON.com>, ""Nemec, ...","Fri, 29 Jun 2001 10:35:17 -0500",FW: ENA Upstream Company information,"John/Gerald,\n\nWe are currently trading under...",0,[]
4,"""Williams, Jason R (Credit)"" <Jason.R.Williams...","""Nemec, Gerald"" <Gerald.Nemec@ENRON.com>, ""Dic...","Fri, 29 Jun 2001 10:40:02 -0500",New Master Physical,Gerald and Stacy -\n\nAttached is a worksheet ...,0,[]


In [15]:
print("Total rows: ",len(df))

Total rows:  3065


In [16]:
df['label'].value_counts()

label
1    1565
0    1500
Name: count, dtype: int64

In [7]:
df.isnull().sum()

sender        2
receiver    113
date          3
subject      50
body          0
label         0
urls          0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

## Data Preprocessing

In [11]:
#replacing nulls with empty strings
df['sender']=df['sender'].fillna('')
df['subject']=df['subject'].fillna('')
df['date']=df['date'].fillna('')

In [12]:
def safe_parse_urls(u):
    if pd.isna(u):
        return []
    return ast.literal_eval(u)

In [13]:
df['urls']=df['urls'].apply(safe_parse_urls)

In [14]:
print("Total rows: ",len(df))

Total rows:  3065


In [18]:
def split_sender(sender_str):
    display_name,email_addr=parseaddr(sender_str)
    domain=email_addr.split("@")[-1].lower() if "@" in email_addr else ""
    return display_name.strip(),domain

In [19]:
df[['sender_display_name','sender_domain']]=df['sender'].apply(
    lambda s:pd.Series(split_sender(s))
)

In [21]:
def row_to_parsed(row):
    return{"subject":row['subject'],
           "body":row['body'],
           "urls":row['urls'],
           "sender_display_name": row['sender_display_name'],
           "sender_domain":row['sender_domain'],
        }